# Vérification des annotations : le texte avant `[` est-il la valeur de l'entité ?

Contrôle de conformité du corpus `all_species_json_final_top20/`. Il n'utilise **que
le corpus** — les fichiers BIO ne servent pas.

**Principe.** L'annotation est réputée juste : sa valeur vient d'Add-my-Pet. Il suffit
donc de vérifier que le texte qui précède immédiatement le `[` énonce **exactement**
cette valeur :

```
... approximately fifty-seven years [am = approximately fifty-seven years]   ✅
... seventeen point five degrees Celsius [T = 17.5 °C]                       ❌
```

Dans le second cas le texte devrait dire `17.5 °C`, tel quel.

La comparaison est faite après normalisation : minuscules, espaces réduits, `°` isolé,
ponctuation de fin ignorée. Une annotation est **conforme** si le texte qui la précède
se termine par sa valeur.

**Annotations imbriquées.** Le corpus en contient (par ex.
`[tp = approximately 60 day at a temperature of 37.5 °C [T = 37.5 °C]s after birth]`).
Une expression régulière naïve du type `\[[^\]]*\]` les compte mal : elle s'arrête au
premier `]` et avale l'annotation interne. Le parseur ci-dessous apparie les crochets.

In [1]:
import json, re, glob, os
from collections import Counter
import pandas as pd

CORPUS_DIR = '../data_annotation/all_species_json_final_top20'
OUT_DIR    = '.'

# Les 20 traits du schéma d'entités
ENTITIES = ['F_R_i','N_R_i','JOi','GSI','Wwi','Wwb','Wwp','Wwx','Ww0',
            'Li','Lb','Lp','am','ab','tp','tx','tR','tg','Ri','T']

print('fichiers corpus :', len(glob.glob(CORPUS_DIR + '/*.json')))

fichiers corpus : 7281


## 1. Parseur d'annotations (appariement de crochets)

In [2]:
START = re.compile(r'\[\s*(' + '|'.join(sorted(ENTITIES, key=len, reverse=True)) + r')\s*=')

def parse_annotations(text):
    """Retourne toutes les annotations, y compris imbriquees.

    Pour chaque '[ENTITY =' trouve, on avance en comptant les crochets jusqu'au ']'
    correspondant. 'nested' signale une annotation malformee (un '[' ouvert a
    l'interieur de la valeur).
    """
    out = []
    for m in START.finditer(text):
        i = m.end(); depth = 1; j = i; nested = False
        while j < len(text) and depth:
            if text[j] == '[':
                depth += 1; nested = True
            elif text[j] == ']':
                depth -= 1
            j += 1
        out.append({'entity': m.group(1),
                    'value': text[i:j-1].strip(),
                    'pos': m.start(),
                    'nested': nested})
    return out

## 2. Comparaison texte ↔ valeur, et classement de la cause

| catégorie | description |
|---|---|
| `malformed` | annotation imbriquée ou tronquée |
| `valeur_en_lettres` | la valeur est en chiffres dans l'annotation, en toutes lettres dans le texte |
| `paraphrase` | le texte énonce autre chose que la valeur annotée |
| `formulation_differente` | la valeur figure dans le texte, mais l'annotation est mal placée ou reformulée |

In [3]:
NUMWORDS = ('one','two','three','four','five','six','seven','eight','nine','ten',
            'eleven','twelve','thirteen','fourteen','fifteen','sixteen','seventeen',
            'eighteen','nineteen','twenty','thirty','forty','fifty','sixty','seventy',
            'eighty','ninety','hundred','thousand','million','point','single','half',
            'annual','biannual','once','twice')

def norm_txt(s):
    """Normalisation pour la comparaison texte <-> valeur."""
    s = s.lower().replace('\u00b0', ' \u00b0 ')
    return re.sub(r'\s+', ' ', s).strip(' .,;:()')

def tail_words(before, value):
    """Les N derniers mots du texte, N = nombre de mots de la valeur."""
    n = len(value.split())
    return ' '.join(before.split()[-n:]) if n else ''

def classify(ann, before):
    if ann['nested']:
        return 'malformed'
    nums = re.findall(r'\d+(?:\.\d+)?', ann['value'])
    if nums and not any(n in before for n in nums):
        return 'valeur_en_lettres' if any(w in before.lower() for w in NUMWORDS) else 'paraphrase'
    return 'formulation_differente'

## 3. Parcours de toutes les espèces

In [4]:
check_rows   = []
per_entity   = Counter()   # annotations totales par entite
total_ann    = 0
total_nested = 0

for f in sorted(glob.glob(CORPUS_DIR + '/*.json')):
    obj = json.load(open(f))
    if isinstance(obj, list):
        obj = obj[0] if obj else {}
    specie = obj.get('specie') or os.path.basename(f)
    text   = obj.get('corrected_text_final') or ''

    for a in parse_annotations(text):
        total_ann += 1
        total_nested += a['nested']
        per_entity[a['entity']] += 1

        before = text[:a['pos']].rstrip()
        if norm_txt(before).endswith(norm_txt(a['value'])):
            continue                                   # conforme

        ctx = before.replace('\n', ' ')[-150:]
        check_rows.append({'specie': specie,
                           'file': os.path.basename(f),
                           'entity': a['entity'],
                           'categorie': classify(a, before[-150:]),
                           'valeur_annotation': a['value'][:150],
                           'texte_avant': tail_words(before, a['value'])[-150:],
                           'nested': a['nested'],
                           'pos': a['pos'],
                           'contexte': ctx})

chk = pd.DataFrame(check_rows)
print('annotations verifiees :', total_ann)
print('dont imbriquees       :', total_nested)
print('non conformes         :', len(chk), f'({100*len(chk)/total_ann:.2f} %)')
print('especes concernees    :', chk['specie'].nunique())
print()
print(chk['categorie'].value_counts().to_string())

annotations verifiees : 119731
dont imbriquees       : 387
non conformes         : 1099 (0.92 %)
especes concernees    : 918

categorie
formulation_differente    406
malformed                 387
valeur_en_lettres         193
paraphrase                113


## 4. Bilan par entité

In [5]:
conf = pd.DataFrame({'annotations': pd.Series(per_entity)}).reindex(ENTITIES).fillna(0).astype(int)
conf['non_conformes'] = [int((chk.entity == e).sum()) for e in ENTITIES]
conf['especes']       = [int(chk[chk.entity == e]['specie'].nunique()) for e in ENTITIES]
conf['taux_%']        = (100 * conf['non_conformes'] / conf['annotations'].replace(0, pd.NA)).round(2)
conf = conf.sort_values('non_conformes', ascending=False)
conf

,annotations,non_conformes,especes,taux_%
T,38982,221,190,0.57
F_R_i,5587,183,174,3.28
tp,5270,120,119,2.28
tx,3535,106,105,3.00
am,9758,98,97,1.00
Ri,2530,80,80,3.16
tR,2348,76,74,3.24
N_R_i,5583,57,54,1.02
Wwi,10443,43,33,0.41
ab,5007,32,32,0.64


In [6]:
# Croisement entite x categorie
pd.crosstab(chk['entity'], chk['categorie']).reindex(ENTITIES).fillna(0).astype(int)

categorie,formulation_differente,malformed,paraphrase,valeur_en_lettres
entity,,,,
F_R_i,65,1,13,104
N_R_i,28,0,7,22
JOi,0,1,0,0
GSI,0,0,0,0
Wwi,40,2,0,1
Wwb,9,2,0,0
Wwp,3,2,0,0
Wwx,0,11,0,0
Ww0,5,0,0,0


## 5. Les espèces en erreur, par entité

In [7]:
sp_err = (chk.groupby(['specie', 'file', 'entity'])
             .size().reset_index(name='non_conformes')
             .sort_values(['non_conformes', 'specie'], ascending=[False, True])
             .reset_index(drop=True))
print('couples (espece, entite) en erreur :', len(sp_err))
print('especes distinctes                 :', sp_err['specie'].nunique())
sp_err.head(30)

couples (espece, entite) en erreur : 1037
especes distinctes                 : 918


,specie,file,entity,non_conformes
0,Octopus_mimus,4654_Octopus_mimus_abj.json,T,5
1,Passerina_amoena,4986_Passerina_amoena_std.json,T,4
2,Hypsypops_rubicundus,3212_Hypsypops_rubicundus_abj.json,T,3
3,Merluccius_merluccius,4029_Merluccius_merluccius_abj.json,T,3
4,Myrmecophaga_tridactyla,4359_Myrmecophaga_tridactyla_stx.json,Wwi,3
5,Notropis_hudsonius,4566_Notropis_hudsonius_abj.json,T,3
6,Alectoris_chukar,0213_Alectoris_chukar_std.json,T,2
7,Ariolimax_columbianus,0563_Ariolimax_columbianus_abj.json,T,2
8,Atherurus_africanus,0643_Atherurus_africanus_stx.json,Wwi,2
9,Bathyraja_maculata,0752_Bathyraja_maculata_std.json,T,2


In [8]:
# Liste des especes concernees, entite par entite
for e in ENTITIES:
    sub = sp_err[sp_err.entity == e]
    if not len(sub):
        print(f'{e:6} : aucune espece en erreur'); continue
    noms = sub.specie.tolist()
    print(f'{e:6} : {len(noms):4d} especes, {int(sub["non_conformes"].sum()):4d} annotations non conformes')
    print('        ', ', '.join(noms[:8]) + (' ...' if len(noms) > 8 else ''))

F_R_i  :  174 especes,  183 annotations non conformes
         Cacicus_chrysonotus, Ceriodaphnia_lacustris, Chelictinia_riocourii, Daphnia_mendotae, Liomys_irroratus, Lipotes_vexillifer, Myotis_septentrionalis, Pheucticus_ludovicianus ...
N_R_i  :   54 especes,   57 annotations non conformes
         Cyclestheria_hislopi, Liomys_irroratus, Tapirus_terrestris, Acrocephalus_melanopogon, Anairetes_parulus, Apodemus_argenteus, Capoeta_capoeta, Carinascincus_ocellatus ...
JOi    :    1 especes,    1 annotations non conformes
         Heliophobius_argenteocinereus
GSI    : aucune espece en erreur
Wwi    :   33 especes,   43 annotations non conformes
         Myrmecophaga_tridactyla, Atherurus_africanus, Cervus_canadensis, Eos_histrio, Macaca_sylvanus, Mellivora_capensis, Negaprion_brevirostris, Okapia_johnstoni ...
Wwb    :   11 especes,   11 annotations non conformes
         Amandava_subflava, Darevskia_derjugini, Emberiza_calandra, Emydura_victoriae, Euoticus_elegantulus, Hierococcyx_spar

## 6. Inspection d'une entité

Changez `ENTITY` pour examiner n'importe lequel des 20 traits.

In [9]:
ENTITY = 'F_R_i'          # <- a modifier
N      = 15               # nombre de cas a afficher

sub = chk[chk.entity == ENTITY]
print(f'{ENTITY} : {len(sub)} annotations non conformes, {sub.specie.nunique()} especes')
print(sub['categorie'].value_counts().to_string()); print()
for _, r in sub.head(N).iterrows():
    print(f"{r.specie}   [{r.categorie}]")
    print(f"   texte      : ...{str(r.contexte)[-110:]}")
    print(f"   annotation : [{r.entity} = {r.valeur_annotation}]")
    print()

F_R_i : 183 annotations non conformes, 174 especes
categorie
valeur_en_lettres         104
formulation_differente     65
paraphrase                 13
malformed                   1

Acipenser_baerii   [formulation_differente]
   texte      : ..._R_i = approximately 1,000,000 eggs per spawning event] combined with a spawning interval of roughly 2.7 years
   annotation : [F_R_i = roughly 1 spawning every 2.7 years]

Acipenser_schrenckii   [formulation_differente]
   texte      : ...ykhtin and Svirskii in 1997, individual females spawn only once every 4 years under natural fluvial conditions
   annotation : [F_R_i = once every 4 years]

Acridotheres_tristis   [valeur_en_lettres]
   texte      : ...es consumed during development. The combination of three clutches of 4.5 eggs each [N_R_i = 4.5 eggs each] and
   annotation : [F_R_i = 3 clutches each year]

Acrocephalus_melanopogon   [valeur_en_lettres]
   texte      : ... output of the Moustached warbler is limited to a single clutch of 4 eg

In [10]:
# Export lisible de l'entite inspectee (tous les cas, pas seulement les N premiers)
path_txt = os.path.join(OUT_DIR, f'{ENTITY}_non_conformes.txt')
with open(path_txt, 'w') as out:
    out.write(f'{ENTITY} — {len(sub)} annotations non conformes\n' + '=' * 100 + '\n\n')
    for i, (_, r) in enumerate(sub.sort_values(['categorie', 'specie']).iterrows(), 1):
        out.write(f"{i:3}. {r.specie}   [{r.categorie}]\n")
        out.write(f"     texte      : ...{str(r.contexte)[-130:]}\n")
        out.write(f"     annotation : [{ENTITY} = {r.valeur_annotation}]\n\n")
sub.to_csv(os.path.join(OUT_DIR, f'{ENTITY}_non_conformes.csv'), index=False)
print('ecrits :', path_txt, 'et', f'{ENTITY}_non_conformes.csv')

ecrits : ./F_R_i_non_conformes.txt et F_R_i_non_conformes.csv


## 7. Export

In [11]:
chk.to_csv(os.path.join(OUT_DIR, 'annotation_text_mismatch.csv'), index=False)
sp_err.to_csv(os.path.join(OUT_DIR, 'species_text_mismatch.csv'), index=False)
conf.to_csv(os.path.join(OUT_DIR, 'conformity_per_entity.csv'))
print('ecrits dans', os.path.abspath(OUT_DIR), ':')
print('  annotation_text_mismatch.csv   chaque annotation non conforme, avec sa cause')
print('  species_text_mismatch.csv      especes x entites en erreur')
print('  conformity_per_entity.csv      taux de non-conformite par entite')

ecrits dans /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/verification_annotation :
  annotation_text_mismatch.csv   chaque annotation non conforme, avec sa cause
  species_text_mismatch.csv      especes x entites en erreur
  conformity_per_entity.csv      taux de non-conformite par entite


## 8. Liste des espèces concernées, sans répétition

Une ligne par espèce (chaque nom apparaît une seule fois), avec le nombre
d'annotations non conformes et la liste des entités posant problème pour cette espèce.
Un second tableau donne la liste des entités concernées, elle aussi sans répétition.

In [12]:
# --- une ligne par espece, sans repetition ---
uniq = (chk.groupby('specie')
          .agg(non_conformes=('entity', 'size'),
               nb_entites=('entity', lambda s: s.nunique()),
               entites=('entity', lambda s: ', '.join(sorted(set(s)))))
          .reset_index()
          .sort_values(['non_conformes', 'specie'], ascending=[False, True])
          .reset_index(drop=True))

print('especes distinctes concernees :', len(uniq))
print('total annotations non conformes :', int(uniq.non_conformes.sum()))
display(uniq.head(25))

# --- entites concernees, sans repetition ---
ents = (chk.groupby('entity')
          .agg(non_conformes=('specie', 'size'),
               especes=('specie', 'nunique'))
          .reset_index()
          .sort_values('non_conformes', ascending=False)
          .reset_index(drop=True))

print()
print('entites concernees :', len(ents), 'sur', len(ENTITIES))
print('entites sans aucun probleme :', sorted(set(ENTITIES) - set(ents.entity)))
display(ents)

# --- export ---
uniq.to_csv(os.path.join(OUT_DIR, 'species_unique_non_conformes.csv'), index=False)
ents.to_csv(os.path.join(OUT_DIR, 'entities_non_conformes.csv'), index=False)
print()
print('ecrits :')
print('  species_unique_non_conformes.csv  une ligne par espece + entites concernees')
print('  entities_non_conformes.csv        une ligne par entite')

especes distinctes concernees : 918
total annotations non conformes : 1099


,specie,non_conformes,nb_entites,entites
0,Myrmecophaga_tridactyla,5,2,"Li, Wwi"
1,Octopus_mimus,5,1,T
2,Passerina_amoena,5,2,"T, tR"
3,Tapirus_terrestris,5,3,"F_R_i, N_R_i, T"
4,Cacicus_chrysonotus,4,2,"F_R_i, T"
5,Liomys_irroratus,4,2,"F_R_i, N_R_i"
6,Macaca_sylvanus,4,2,"Li, Wwi"
7,Negaprion_brevirostris,4,2,"Wwi, Wwp"
8,Varecia_variegata,4,4,"F_R_i, N_R_i, T, tp"
9,Acrocephalus_melanopogon,3,3,"F_R_i, N_R_i, tx"



entites concernees : 19 sur 20
entites sans aucun probleme : ['GSI']


,entity,non_conformes,especes
0,T,221,190
1,F_R_i,183,174
2,tp,120,119
3,tx,106,105
4,am,98,97
5,Ri,80,80
6,tR,76,74
7,N_R_i,57,54
8,Wwi,43,33
9,ab,32,32



ecrits :
  species_unique_non_conformes.csv  une ligne par espece + entites concernees
  entities_non_conformes.csv        une ligne par entite


## 9. Copie des fichiers JSON des espèces concernées

Copie, dans un dossier `species_non_conformes_json/` créé à côté du notebook, le
fichier source de chacune des espèces présentant au moins une annotation non conforme.
Les originaux de `all_species_json_final_top20/` ne sont pas modifiés : ce dossier est
un espace de travail pour la correction.

In [13]:
import shutil

DEST = os.path.join(OUT_DIR, 'species_non_conformes_json')
os.makedirs(DEST, exist_ok=True)

fichiers = sorted(chk['file'].unique())
copies, manquants = 0, []
for fn in fichiers:
    src = os.path.join(CORPUS_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DEST, fn)); copies += 1
    else:
        manquants.append(fn)

print('especes concernees :', len(fichiers))
print('fichiers copies    :', copies)
if manquants:
    print('introuvables       :', len(manquants), manquants[:5])
print('destination        :', os.path.abspath(DEST))

# recapitulatif ecrit a cote des fichiers copies
uniq.to_csv(os.path.join(DEST, '_recap_especes.csv'), index=False)
chk.to_csv(os.path.join(DEST, '_recap_annotations.csv'), index=False)
print('recapitulatifs     : _recap_especes.csv, _recap_annotations.csv')

especes concernees : 918
fichiers copies    : 918
destination        : /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/verification_annotation/species_non_conformes_json
recapitulatifs     : _recap_especes.csv, _recap_annotations.csv


## 10. Correction des annotations `malformed`

Répare, **dans `species_non_conformes_json/`**, les annotations où le fragment
`at a temperature of X °C [T = X °C]` a été inséré à l'intérieur d'une autre annotation,
coupant un mot en deux :

```
avant : [am = approximate at a temperature of 40.2 °C [T = 40.2 °C]ly 6,424 days]
après : [am = approximately 6,424 days] at a temperature of 40.2 °C [T = 40.2 °C]
```

Trois opérations déterministes : retirer le fragment (ce qui recolle le mot coupé),
normaliser les espaces, replacer le fragment après le crochet fermant. Les températures
répétées à l'identique sont dédupliquées (17 annotations portent deux fois le même
fragment).

⚠️ Cette cellule **écrit** dans `species_non_conformes_json/`. Les fichiers d'origine
de `all_species_json_final_top20/` ne sont pas touchés. Pour repartir de zéro, relancer
la section 9.

In [14]:
DEST = os.path.join(OUT_DIR, 'species_non_conformes_json')
FRAG = re.compile(r'\s*at a temperature of\s*([\d.]+)\s*°C\s*\[T\s*=\s*([\d.]+)\s*°C\]')

def annotations_with_span(text):
    """Comme parse_annotations, mais renvoie aussi la position de fin."""
    out = []
    for m in START.finditer(text):
        i = m.end(); depth = 1; j = i; nested = False
        while j < len(text) and depth:
            if text[j] == '[':
                depth += 1; nested = True
            elif text[j] == ']':
                depth -= 1
            j += 1
        out.append({'entity': m.group(1), 'value': text[i:j-1].strip(),
                    'start': m.start(), 'end': j, 'nested': nested})
    return out

def repair_text(text):
    """Repare les annotations malformees. Renvoie (texte, journal).

    Le fragment insere peut avoir pris la place d'un espace ('up to 10|years')
    ou avoir coupe un mot ('approximate|ly'). On ne devine pas : on construit les
    deux reconstructions et on retient celle qui correspond a la prose qui precede.
    """
    journal = []
    casses = [a for a in annotations_with_span(text) if a['nested']]
    for a in reversed(casses):          # de la fin vers le debut : les offsets restent valides
        temps = [t[0] for t in FRAG.findall(a['value'])]
        if not temps:
            continue                    # imbrication d'une autre nature : on ne touche pas
        colle  = re.sub(r'\s{2,}', ' ', FRAG.sub('',  a['value'])).strip()
        espace = re.sub(r'\s{2,}', ' ', FRAG.sub(' ', a['value'])).strip()
        prose  = norm_txt(text[:a['start']].rstrip())
        if   prose.endswith(norm_txt(colle)):  clean, variante = colle,  'recolle'
        elif prose.endswith(norm_txt(espace)): clean, variante = espace, 'espace_restitue'
        else:                                  clean, variante = colle,  'recolle_non_confirme'
        vus = []
        for t in temps:                 # deduplication en conservant l'ordre
            if t not in vus:
                vus.append(t)
        suffixe = ''.join(f' at a temperature of {t} °C [T = {t} °C]' for t in vus)
        nouveau = f"[{a['entity']} = {clean}]{suffixe}"
        journal.append({'entity': a['entity'], 'variante': variante,
                        'avant': text[a['start']:a['end']],
                        'apres': nouveau, 'fragments': len(temps)})
        text = text[:a['start']] + nouveau + text[a['end']:]
    return text, journal


journal_global, fichiers_modifies = [], 0
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    repare, journal = repair_text(texte)
    if not journal:
        continue
    obj['corrected_text_final'] = repare
    with open(path, 'w') as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2)
    fichiers_modifies += 1
    for j in journal:
        j['specie'] = obj.get('specie'); j['file'] = os.path.basename(path)
        journal_global.append(j)

rep = pd.DataFrame(journal_global)
print('fichiers modifies      :', fichiers_modifies)
print('annotations reparees   :', len(rep))
if len(rep):
    print('dont a deux fragments  :', int((rep.fragments > 1).sum()))
    print(); print('variante de reconstruction :'); print(rep.variante.value_counts().to_string())
    print(); print('par entite :'); print(rep.entity.value_counts().to_string())
    rep.to_csv(os.path.join(OUT_DIR, 'reparations_malformed.csv'), index=False)
    print(); print('journal ecrit : reparations_malformed.csv')

fichiers modifies      : 346
annotations reparees   : 387
dont a deux fragments  : 17

variante de reconstruction :
variante
recolle                 359
espace_restitue          27
recolle_non_confirme      1

par entite :
entity
tp       105
tx       101
am        70
tR        57
ab        24
Wwx       11
tg         6
Lp         4
Wwi        2
Wwb        2
Wwp        2
T          1
JOi        1
F_R_i      1

journal ecrit : reparations_malformed.csv


In [15]:
# --- verification apres reparation ---
restants, non_conformes_apres, total_apres = 0, 0, 0
exemples = []
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    for a in parse_annotations(texte):
        total_apres += 1
        if a['nested']:
            restants += 1
            if len(exemples) < 5:
                exemples.append((obj.get('specie'), a['entity'], a['value'][:100]))
        before = texte[:a['pos']].rstrip()
        if not norm_txt(before).endswith(norm_txt(a['value'])):
            non_conformes_apres += 1

print('--- apres reparation, sur les', len(glob.glob(DEST + "/*.json")) - 2, 'fichiers du dossier ---')
print('annotations totales      :', total_apres)
print('annotations malformees   :', restants, '(attendu : 0)')
print('annotations non conformes:', non_conformes_apres)
print('rappel avant reparation  : 1099 non conformes, dont 387 malformees')
for e in exemples:
    print('   restant :', e)

--- apres reparation, sur les 916 fichiers du dossier ---
annotations totales      : 16969
annotations malformees   : 0 (attendu : 0)
annotations non conformes: 713
rappel avant reparation  : 1099 non conformes, dont 387 malformees


## 11. Corrections scriptables

Trois actions déterministes, sans modèle de langue et sans décision éditoriale :

| action | principe |
|---|---|
| `parenthese` | retirer la parenthèse insérée en plein mot, recoller |
| `deplacer`   | la valeur est dans la phrase : déplacer le crochet juste après elle |
| `supprimer`  | la valeur est absente **et** l'espèce garde une autre mention conforme du même trait |

Ne sont **pas** traités ici les cas qui demandent un arbitrage : voir la section 12.

⚠️ `DRY_RUN = True` par défaut : rien n'est écrit, le plan de correction est seulement
affiché et exporté. Passer à `False` pour appliquer.

In [16]:
DRY_RUN = True          # <- passer a False pour ecrire dans les fichiers
DEST = os.path.join(OUT_DIR, 'species_non_conformes_json')
PAREN = re.compile(r'\(([^()]*)\)')

FIN_PHRASE = re.compile(r'(?<!\d)\.(?!\d)')      # un point qui n'est pas une decimale

def phrase_bounds(t, pos):
    s = 0
    for m in FIN_PHRASE.finditer(t, 0, pos):
        s = m.end()
    m = FIN_PHRASE.search(t, pos)
    return s, (m.start() if m else len(t))

def prose_seule(texte, s, e, annots):
    """Phrase [s,e) privee de toutes ses annotations.

    Renvoie (texte_prose, idx) ou idx[k] est la position, dans le texte d'origine,
    du k-ieme caractere de la prose. Indispensable : chercher la valeur dans le texte
    brut la trouverait a l'interieur d'autres annotations, ce qui n'est pas de la prose.
    """
    keep, idx = [], []
    spans = sorted((a['start'], a['end']) for a in annots if s <= a['start'] < e)
    i = s
    for st, en in spans:
        while i < st:
            keep.append(texte[i]); idx.append(i); i += 1
        i = max(i, en)
    while i < e:
        keep.append(texte[i]); idx.append(i); i += 1
    return ''.join(keep), idx

def motif(valeur):
    """Regex tolerante aux espaces pour retrouver la valeur dans la prose."""
    mots = [re.escape(w) for w in valeur.split()]
    return re.compile(r'\s*'.join(mots), re.I) if mots else None

def plan_fichier(texte, conformes):
    """Liste des actions a appliquer pour un fichier."""
    actions = []
    annots = annotations_with_span(texte)
    for a in annots:
        brut = texte[a['start']:a['end']]
        if norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(a['value'])):
            continue                                    # deja conforme

        # --- 1. parenthese inseree en plein mot ---------------------------
        m = PAREN.search(a['value'])
        if m:
            interne = m.group(1).strip()
            colle   = re.sub(r'\s{2,}', ' ', PAREN.sub('', a['value'])).strip()
            # dans 14 cas sur 15 le contenu de la parenthese EST la valeur correcte
            valeur = interne if norm_txt(interne) == norm_txt(colle.replace(' ', '')) or len(interne) >= len(colle) else colle
            actions.append({'action': 'parenthese', 'entity': a['entity'],
                            'start': a['start'], 'end': a['end'], 'avant': brut,
                            'apres': f"[{a['entity']} = {valeur}]"})
            continue

        s, e = phrase_bounds(texte, a['start'])
        prose, idx = prose_seule(texte, s, e, annots)
        pat = motif(a['value'])

        # --- 2. la valeur est dans la PROSE : deplacer le crochet ---------
        cible = None
        if pat:
            for mm in pat.finditer(prose):
                cible = mm
        if cible is not None and cible.end() <= len(idx):
            actions.append({'action': 'deplacer', 'entity': a['entity'],
                            'start': a['start'], 'end': a['end'],
                            'insert_at': idx[cible.end() - 1] + 1,
                            'avant': brut, 'apres': brut})
            continue

        # --- 3. valeur absente, mais le trait est annote ailleurs ---------
        if conformes.get(a['entity'], 0) > 0:
            actions.append({'action': 'supprimer', 'entity': a['entity'],
                            'start': a['start'], 'end': a['end'],
                            'avant': brut, 'apres': ''})
    return actions


plan, fichiers = [], 0
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    conformes = Counter(a['entity'] for a in parse_annotations(texte)
                        if norm_txt(texte[:a['pos']].rstrip()).endswith(norm_txt(a['value'])))
    actions = plan_fichier(texte, conformes)
    if not actions:
        continue
    fichiers += 1
    for act in actions:
        plan.append({**{k: v for k, v in act.items() if k not in ('start', 'end', 'insert_at')},
                     'specie': obj.get('specie'), 'file': os.path.basename(path)})

    if not DRY_RUN:
        for act in sorted(actions, key=lambda x: -x['start']):
            if act['action'] == 'deplacer':
                bloc = texte[act['start']:act['end']]
                ins  = act['insert_at']
                if ins < act['start']:
                    texte = (texte[:ins] + ' ' + bloc
                             + texte[ins:act['start']].rstrip() + texte[act['end']:])
                else:
                    texte = (texte[:act['start']].rstrip() + texte[act['end']:ins]
                             + ' ' + bloc + texte[ins:])
            elif act['action'] == 'supprimer':
                texte = texte[:act['start']].rstrip() + texte[act['end']:]
            else:
                texte = texte[:act['start']] + act['apres'] + texte[act['end']:]
        obj['corrected_text_final'] = texte
        with open(path, 'w') as fh:
            json.dump(obj, fh, ensure_ascii=False, indent=2)

pl = pd.DataFrame(plan)
print('MODE :', 'SIMULATION (rien ecrit)' if DRY_RUN else 'APPLIQUE')
print('fichiers concernes :', fichiers, '| actions :', len(pl))
if len(pl):
    print(); print(pl.action.value_counts().to_string())
    print(); print(pd.crosstab(pl.entity, pl.action).to_string())
    pl.to_csv(os.path.join(OUT_DIR, 'plan_corrections_scriptables.csv'), index=False)
    print(); print('plan exporte : plan_corrections_scriptables.csv')

MODE : SIMULATION (rien ecrit)
fichiers concernes : 467 | actions : 549

action
supprimer     272
deplacer      262
parenthese     15

action  deplacer  parenthese  supprimer
entity                                 
F_R_i         20           0         63
Lb             5           0          2
Li            12           0          7
Lp             8           0          4
N_R_i         16           0         22
Ri            38           0         14
T            100           0        115
Ww0            4           0          1
Wwb            7           0          2
Wwi           27           2         11
Wwp            2           0          1
ab             5           1          1
am            11           4         13
tR             3           4          4
tg             0           1          1
tp             4           3          6
tx             0           0          5

plan exporte : plan_corrections_scriptables.csv


## 12. Ce qui demande un arbitrage humain

Volontairement hors de la section 11. Trois familles, exportées pour décision :

| famille | nature | pourquoi pas de script |
|---|---|---|
| `A_equivalence_semantique` | « a single clutch each year » vs `1 clutch each year` | la prose est **correcte**, et meilleure que l'annotation. Réécrire dégraderait le corpus pour satisfaire un test. Alternative : assouplir la comparaison (`single`≡1, `half`≡0.5) |
| `D3_forme_reciproque` | « every 2 days » vs `0.5 per day` | il faut soit ajouter une apposition (le texte s'éloigne du texte réel), soit aligner l'annotation sur la prose (la garantie « verbatim AmP » doit être reformulée) |
| `T_cadre_partage` | « under the same thermal conditions » vs `[T = 17 °C]` | vient d'un conflit dans le prompt (lignes 58 et 61). Décider d'abord la règle, corriger ensuite |

À traiter **après** l'audit des valeurs dérivées (`F_R_i` / `Ri` contre AmP) : corriger la
forme d'une valeur fausse ne ferait que la rendre plus visible.

In [17]:
EQUIV = ('single', 'half', 'once', 'twice', 'annual', 'biannual', 'a pair', 'one')

arbitrage = []
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    conformes = Counter(a['entity'] for a in parse_annotations(texte)
                        if norm_txt(texte[:a['pos']].rstrip()).endswith(norm_txt(a['value'])))
    traitees = {(x['start'], x['end']) for x in plan_fichier(texte, conformes)}

    for a in annotations_with_span(texte):
        if norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(a['value'])):
            continue
        if (a['start'], a['end']) in traitees:
            continue                                    # deja pris en section 11
        s, e = phrase_bounds(texte, a['start'])
        ph = texte[s:e]
        if a['entity'] == 'T':
            fam = 'T_cadre_partage'
        elif any(w in ph.lower() for w in EQUIV):
            fam = 'A_equivalence_semantique'
        else:
            fam = 'D3_forme_reciproque'
        arbitrage.append({'famille': fam, 'specie': obj.get('specie'),
                          'file': os.path.basename(path), 'entity': a['entity'],
                          'valeur_annotation': a['value'][:120],
                          'prose': re.sub(r'\s+', ' ', ph).strip()[-200:]})

arb = pd.DataFrame(arbitrage)
print('cas demandant un arbitrage :', len(arb), '| especes :', arb.specie.nunique() if len(arb) else 0)
if len(arb):
    print(); print(arb.famille.value_counts().to_string())
    print(); print(pd.crosstab(arb.entity, arb.famille).to_string())
    arb.to_csv(os.path.join(OUT_DIR, 'cas_arbitrage_humain.csv'), index=False)
    print(); print('exporte : cas_arbitrage_humain.csv')

cas demandant un arbitrage : 164 | especes : 147

famille
A_equivalence_semantique    115
D3_forme_reciproque          44
T_cadre_partage               5

famille  A_equivalence_semantique  D3_forme_reciproque  T_cadre_partage
entity                                                                 
F_R_i                          82                   18                0
N_R_i                          14                    5                0
Ri                             16                   12                0
T                               0                    0                5
Wwi                             0                    1                0
ab                              0                    1                0
tR                              3                    5                0
tp                              0                    2                0

exporte : cas_arbitrage_humain.csv


---
# 13. Application des corrections scriptables

⚠️ Écrit **uniquement** dans `species_non_conformes_json/`. Le dossier source
`all_species_json_final_top20/` n'est jamais ouvert en écriture.

Deux actions seulement, toutes deux sans perte d'information :

| action | principe |
|---|---|
| `parenthese` | retirer la parenthèse insérée en plein mot ; la valeur correcte est son contenu |
| `deplacer`   | la valeur figure dans la prose : déplacer le crochet juste après elle |

**La suppression n'est plus appliquée ici.** Depuis la décision de faire porter à
l'annotation le texte de la prose, une annotation dont la valeur est absente n'est plus
forcément à jeter : si la prose exprime le trait sous une autre forme, elle doit être
réannotée. Ces cas partent tous en section 14.

In [18]:
APPLIQUER = True   # deja applique : relancer la section 9 pour repartir des originaux        # False pour simuler

applique, fichiers = [], 0
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    conformes = Counter(a['entity'] for a in parse_annotations(texte)
                        if norm_txt(texte[:a['pos']].rstrip()).endswith(norm_txt(a['value'])))
    actions = [a for a in plan_fichier(texte, conformes)
               if a['action'] in ('parenthese', 'deplacer')]
    if not actions:
        continue
    fichiers += 1
    for act in sorted(actions, key=lambda x: -x['start']):
        avant_txt = texte
        if act['action'] == 'deplacer':
            bloc, ins = texte[act['start']:act['end']], act['insert_at']
            if ins < act['start']:
                texte = (texte[:ins] + ' ' + bloc
                         + texte[ins:act['start']].rstrip() + texte[act['end']:])
            else:
                texte = (texte[:act['start']].rstrip() + texte[act['end']:ins]
                         + ' ' + bloc + texte[ins:])
        else:
            texte = texte[:act['start']] + act['apres'] + texte[act['end']:]
        applique.append({'specie': obj.get('specie'), 'file': os.path.basename(path),
                         'action': act['action'], 'entity': act['entity'],
                         'avant': act['avant'], 'apres': act['apres']})
    if APPLIQUER:
        obj['corrected_text_final'] = texte
        with open(path, 'w') as fh:
            json.dump(obj, fh, ensure_ascii=False, indent=2)

ap = pd.DataFrame(applique)
print('MODE :', 'APPLIQUE' if APPLIQUER else 'SIMULATION')
print('fichiers touches :', fichiers, '| actions :', len(ap))
if len(ap):
    print(); print(ap.action.value_counts().to_string())
    ap.to_csv(os.path.join(OUT_DIR, 'corrections_appliquees.csv'), index=False)

# --- verification ---
tot = nc = nested = 0
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    t = obj.get('corrected_text_final') or ''
    for a in parse_annotations(t):
        tot += 1; nested += a['nested']
        if not norm_txt(t[:a['pos']].rstrip()).endswith(norm_txt(a['value'])):
            nc += 1
print(); print('--- apres application ---')
print('annotations :', tot, '| malformees :', nested, '| non conformes :', nc)

MODE : APPLIQUE
fichiers touches : 236 | actions : 277

action
deplacer      262
parenthese     15



--- apres application ---
annotations : 16969 | malformees : 0 | non conformes : 486


---
# 14. Cas à réannoter : l'annotation doit reprendre la prose

**Règle retenue** : l'annotation ne doit pas contenir un calcul, mais le texte tel qu'il
est écrit.

```
prose      : « ... and spawn approximately every 2 days »
avant      : [F_R_i = 0.5 spawning events per day]      <- valeur derivee, absente du texte
apres      : [F_R_i = approximately every 2 days]        <- le texte lui-meme
```

Le span proposé ci-dessous est calculé automatiquement : on prend, dans la phrase
débarrassée de ses annotations, le groupe de mots qui précède immédiatement le crochet,
borné par la ponctuation ou une conjonction. **C'est une proposition, pas une décision** —
la délimitation exacte d'un span demande un jugement, d'où la colonne `span_propose`
à valider (par vous ou par un modèle) avant application en section 15.

In [19]:
TEMP = re.compile(r'\d+(?:\.\d+)?\s*°C')
BORNES = re.compile(r'\b(and|with|that|which|while|but|as|of|in|at|for|from|to|by|is|are|was|were)\b', re.I)
CHIFFRE = re.compile(r'\d')

def span_candidat(prose):
    """Groupe de mots en fin de prose, borne par la ponctuation ou une conjonction."""
    seg = re.sub(r'\s+', ' ', prose).rstrip()
    cut = max(seg.rfind(c) for c in ';,:')
    if cut > 0:
        seg = seg[cut+1:]
    b = list(BORNES.finditer(seg))
    if b:
        seg = seg[b[-1].end():]
    return seg.strip()

reannoter, a_supprimer = [], []
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    annots = annotations_with_span(texte)
    for a in annots:
        if norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(a['value'])):
            continue
        s, e = phrase_bounds(texte, a['start'])
        prose_phrase, _ = prose_seule(texte, s, e, annots)
        prose_avant, _  = prose_seule(texte, s, a['start'], annots)
        ligne = {'specie': obj.get('specie'), 'file': os.path.basename(path),
                 'entity': a['entity'], 'valeur_actuelle': a['value'][:120],
                 'prose': re.sub(r'\s+', ' ', prose_phrase).strip()[:260], 'pos': a['start']}

        # T dont la phrase ne contient aucune temperature -> a supprimer
        if a['entity'] == 'T' and not TEMP.search(prose_phrase):
            a_supprimer.append({**ligne, 'motif': 'aucune temperature dans la phrase'})
            continue
        # prose sans le moindre chiffre -> rien a annoter
        cand = span_candidat(prose_avant)
        if not CHIFFRE.search(cand):
            a_supprimer.append({**ligne, 'motif': 'aucune valeur chiffree avant le crochet'})
            continue
        reannoter.append({**ligne, 'span_propose': cand[:120]})

ra  = pd.DataFrame(reannoter)
sup = pd.DataFrame(a_supprimer)
print('a REANNOTER  :', len(ra),  '| especes :', ra.specie.nunique()  if len(ra)  else 0)
print('a SUPPRIMER  :', len(sup), '| especes :', sup.specie.nunique() if len(sup) else 0)
if len(sup):
    print(); print(sup.motif.value_counts().to_string())
    sup.to_csv(os.path.join(OUT_DIR, 'a_supprimer.csv'), index=False)
if len(ra):
    print(); print(ra.entity.value_counts().to_string())
    ra.to_csv(os.path.join(OUT_DIR, 'a_reannoter.csv'), index=False)
    print(); print('exportes : a_reannoter.csv (span_propose a valider), a_supprimer.csv')
ra.head(15)

a REANNOTER  : 170 | especes : 157
a SUPPRIMER  : 316 | especes : 277

motif
aucune valeur chiffree avant le crochet    205
aucune temperature dans la phrase          111

entity
F_R_i    60
T        32
Wwi      15
N_R_i    11
Ri       10
tp        7
Li        7
am        7
tR        5
tx        4
Lp        4
tg        2
Lb        2
ab        2
Ww0       1
Wwb       1

exportes : a_reannoter.csv (span_propose a valider), a_supprimer.csv


,specie,file,entity,valeur_actuelle,prose,pos,span_propose
0,Acanthogobius_flavimanus,0039_Acanthogobius_flavimanus_std.json,T,17.9 °C,The maximum daily spawning rate of approximate...,5497,17.9 °C
1,Acipenser_baerii,0073_Acipenser_baerii_std.json,F_R_i,roughly 1 spawning every 2.7 years,"Under optimal conditions at 15 °C , the maximu...",3501,roughly 2.7 years
2,Acrocephalus_melanopogon,0100_Acrocephalus_melanopogon_std.json,N_R_i,4 eggs per clutch,The reproductive output of the Moustached warb...,4389,4 eggs annually
3,Acrocephalus_melanopogon,0100_Acrocephalus_melanopogon_std.json,F_R_i,1 clutch each year,The reproductive output of the Moustached warb...,4417,4 eggs annually
4,Acroperus_harpae,0107_Acroperus_harpae_std.json,F_R_i,approximately 7 days,"Under continued exposure to 12 °C , the sustai...",5460,a 7-day intermolt interval
5,Aerodramus_brevirostris,0135_Aerodramus_brevirostris_std.json,F_R_i,1 clutch each year,Under the same thermal regime of 38.1 °C that ...,3523,2 eggs per clutch each year
6,Aethopyga_siparaja,0153_Aethopyga_siparaja_std.json,T,41.6 °C,at a temperature of 41.6 °C at a temperature o...,5721,41.6 °C
7,Afrotis_afra,0158_Afrotis_afra_std.json,Ww0,52-64 grams,Reported values for neonatal mass in this spec...,3267,64 grams
8,Alcelaphus_buselaphus,0207_Alcelaphus_buselaphus_stx.json,T,37.8 °C,at a temperature of 37 at a temperature of 37....,4749,37.8 °C
9,Alectoris_chukar,0213_Alectoris_chukar_std.json,T,approximately 40.7 °C,"This pubertal transition, occurring under typi...",2376,occurring under typical thermal regimes around...


---
# 15. Réannotation assistée par modèle

La colonne `span_propose` de la section 14 n'est qu'une heuristique, et elle se trompe
souvent : elle tronque (`52-64 grams` → `64 grams`) ou attrape la valeur d'une entité
voisine. **Délimiter un span est un jugement, pas un calcul.**

Cette section prépare le travail pour un modèle, puis réinjecte ses réponses avec
vérification automatique. Le garde-fou est simple : le span rendu doit être une
**sous-chaîne littérale** de la prose qui précède le crochet. Toute réponse qui échoue
à ce test est rejetée, pas appliquée.

### Consigne à donner au modèle

> Voici une phrase d'un corpus biologique et une annotation d'entité. L'annotation porte
> actuellement une valeur *calculée* (issue de la base Add-my-Pet) qui n'apparaît pas
> telle quelle dans la phrase. Rends le fragment **exact** de la phrase qui exprime ce
> trait, recopié caractère pour caractère, sans rien ajouter ni reformuler. Si aucun
> fragment de la phrase n'exprime ce trait, réponds `NONE`.

Le fichier `llm_taches.jsonl` contient une tâche par ligne, prête à envoyer.

In [20]:
# --- preparation des taches ---
taches = []
for _, r in ra.iterrows():
    taches.append({'specie': r['specie'], 'file': r['file'], 'pos': int(r['pos']),
                   'entity': r['entity'],
                   'valeur_amp': r['valeur_actuelle'],
                   'phrase': r['prose'],
                   'indice': r['span_propose']})

chemin = os.path.join(OUT_DIR, 'llm_taches.jsonl')
with open(chemin, 'w') as fh:
    for t in taches:
        fh.write(json.dumps(t, ensure_ascii=False) + '\n')
print('taches ecrites :', len(taches), '->', chemin)
print()
print('exemple de tache :')
print(json.dumps(taches[0], ensure_ascii=False, indent=2)[:600])

taches ecrites : 170 -> ./llm_taches.jsonl

exemple de tache :
{
  "specie": "Acanthogobius_flavimanus",
  "file": "0039_Acanthogobius_flavimanus_std.json",
  "pos": 5497,
  "entity": "T",
  "valeur_amp": "17.9 °C",
  "phrase": "The maximum daily spawning rate of approximately 101 eggs per event was recorded exclusively at 17.9 °C , a temperature that simultaneously governs the maximum lifespan of approximately 4.5 years under identical thermal conditions",
  "indice": "17.9 °C"
}


In [21]:
# --- application des reponses validees ---
# Attend un CSV 'llm_reponses.csv' avec les colonnes : file, pos, entity, span
# (span = fragment exact recopie de la phrase, ou NONE)

REPONSES = os.path.join(OUT_DIR, 'llm_reponses.csv')

if not os.path.exists(REPONSES):
    print('aucun fichier', REPONSES, '- rien a appliquer.')
    print('Deposez-y les reponses du modele puis relancez cette cellule.')
else:
    rep = pd.read_csv(REPONSES)
    applique, rejete = [], []
    for fichier, grp in rep.groupby('file'):
        path = os.path.join(DEST, fichier)
        obj = json.load(open(path))
        texte = obj['corrected_text_final']
        for _, r in grp.sort_values('pos', ascending=False).iterrows():
            span = str(r['span']).strip()
            annots = annotations_with_span(texte)
            cible = next((a for a in annots if a['start'] == int(r['pos'])), None)
            if cible is None:
                rejete.append({**dict(r), 'motif': 'annotation introuvable a cette position'}); continue
            if span.upper() == 'NONE':
                rejete.append({**dict(r), 'motif': 'NONE - a traiter par suppression'}); continue
            # GARDE-FOU : le span doit terminer la prose qui precede le crochet
            if not norm_txt(texte[:cible['start']].rstrip()).endswith(norm_txt(span)):
                rejete.append({**dict(r), 'motif': 'span absent de la prose qui precede'}); continue
            nouveau = f"[{cible['entity']} = {span}]"
            applique.append({'file': fichier, 'entity': cible['entity'],
                             'avant': texte[cible['start']:cible['end']], 'apres': nouveau})
            texte = texte[:cible['start']] + nouveau + texte[cible['end']:]
        obj['corrected_text_final'] = texte
        with open(path, 'w') as fh:
            json.dump(obj, fh, ensure_ascii=False, indent=2)
    print('reannotations appliquees :', len(applique))
    print('rejetees                 :', len(rejete))
    if rejete:
        print(); print(pd.DataFrame(rejete).motif.value_counts().to_string())
        pd.DataFrame(rejete).to_csv(os.path.join(OUT_DIR, 'llm_reponses_rejetees.csv'), index=False)
    if applique:
        pd.DataFrame(applique).to_csv(os.path.join(OUT_DIR, 'reannotations_appliquees.csv'), index=False)

aucun fichier ./llm_reponses.csv - rien a appliquer.
Deposez-y les reponses du modele puis relancez cette cellule.


---
# 16. `valeur_en_lettres` — la part scriptable

Règle : l'annotation reprend le texte. Ici le texte écrit le nombre en toutes lettres.

```
prose : « ... females generally produce just over one litter per year »
avant : [F_R_i = 1.36 litters per year]      <- valeur calculee, absente du texte
apres : [F_R_i = just over one litter per year]
```

**Un cas n'est traité que s'il passe les sept garde-fous suivants** — sinon il part
vers la section 15 (modèle), sans être touché :

1. l'annotation est non conforme et non malformée ;
2. sa valeur contient un chiffre qui n'apparaît pas dans la prose ;
3. un mot-nombre figure dans la prose juste avant le crochet ;
4. le span va de ce mot-nombre (modificateurs inclus) **jusqu'au crochet** — la conformité
   est donc acquise par construction ;
5. le span fait entre 2 et 9 mots ;
6. il ne se termine pas par un mot outil (`and`, `of`, `the`, `to`…) ;
7. la valeur numérique du span correspond à celle de l'annotation (tolérance 6 %).

Écrit uniquement dans `species_non_conformes_json/`.

In [22]:
APPLIQUER_16 = True         # passer a True pour ecrire

MOTS_NB = {'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,'eight':8,'nine':9,
           'ten':10,'eleven':11,'twelve':12,'thirteen':13,'fourteen':14,'fifteen':15,
           'sixteen':16,'seventeen':17,'eighteen':18,'nineteen':19,'twenty':20,'thirty':30,
           'forty':40,'fifty':50,'sixty':60,'seventy':70,'eighty':80,'ninety':90,
           'hundred':100,'thousand':1000,'million':1000000,'half':0.5,'single':1,
           'once':1,'twice':2}
MODIF   = ('about','approximately','roughly','around','nearly','over','just','almost','up','to',
           'a','an','the','least','most','more','than','fewer','some')
OUTILS  = ('and','or','of','the','a','an','to','with','in','at','for','from','by','that','which',
           'is','are','was','were','as','on')
MOT_NB  = re.compile(r'\b(' + '|'.join(sorted(MOTS_NB, key=len, reverse=True)) + r')\b', re.I)

def valeur_mots(txt):
    """Valeur numerique d'un groupe de mots-nombres ('two to three' -> 3)."""
    toks = [t.lower() for t in re.findall(r'[A-Za-z]+', txt) if t.lower() in MOTS_NB]
    if not toks:
        return None
    tot = cur = 0
    for t in toks:
        v = MOTS_NB[t]
        if v == 100:
            cur = (cur or 1) * 100
        elif v in (1000, 1000000):
            tot += (cur or 1) * v; cur = 0
        else:
            cur += v
    return tot + cur

def span_lettres(prose_avant, valeur_annotee, nombres_voisins=()):
    """Renvoie (span, motif_de_rejet). span=None si un garde-fou echoue.

    'nombres_voisins' : les nombres portes par les autres annotations de la phrase.
    Un span qui les contient melange deux entites et doit etre rejete.
    """
    nums = re.findall(r'\d+(?:\.\d+)?', valeur_annotee)
    if not nums:
        return None, 'annotation sans chiffre'
    ms = list(MOT_NB.finditer(prose_avant))
    if not ms:
        return None, 'aucun mot-nombre dans la prose'
    # remonter au debut du groupe : dernier mot-nombre, puis ses modificateurs a gauche
    k = ms[-1].start()
    for m in reversed(ms[:-1]):                       # chaine 'two to three'
        entre = prose_avant[m.end():k].strip(' ,-')
        if entre.lower() in ('', 'to', 'and', 'or', '-'):
            k = m.start()
        else:
            break
    gauche = prose_avant[:k].split()
    while gauche and gauche[-1].lower().strip(',') in MODIF:
        k -= len(gauche[-1]) + 1
        gauche.pop()
    span = re.sub(r'\s+', ' ', prose_avant[max(k, 0):]).strip(' ,;:')
    # ne pas commencer par un mot outil ou un article
    mots_g = span.split()
    while mots_g and mots_g[0].lower().strip(',') in OUTILS:
        mots_g.pop(0)
    span = ' '.join(mots_g)
    if '[' in span or ']' in span:
        return None, 'span contient un crochet'
    mots = span.split()
    if not (2 <= len(mots) <= 9):
        return None, f'span de {len(mots)} mots'
    if mots[-1].lower().strip('.,') in OUTILS:
        return None, 'span termine par un mot outil'
    for n in nombres_voisins:
        if re.search(r'(?<![\d.])' + re.escape(n) + r'(?![\d])', span):
            return None, f'span contient {n}, valeur d\'une autre entite'
    vm, va = valeur_mots(span), float(nums[0])
    if vm is None:
        return None, 'span sans valeur calculable'
    if abs(vm - va) > max(0.6, 0.06 * abs(va)):
        return None, f'valeur du span ({vm}) != annotation ({va})'
    return span, None


faits, laisses = [], []
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    modifs = []
    for a in annotations_with_span(texte):
        if a['nested'] or norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(a['value'])):
            continue
        s, _ = phrase_bounds(texte, a['start'])
        prose_avant, _ = prose_seule(texte, s, a['start'], annotations_with_span(texte))
        nums = re.findall(r'\d+(?:\.\d+)?', a['value'])
        if not nums or any(n in prose_avant[-150:] for n in nums):
            continue                                   # pas un cas 'valeur en lettres'
        if not MOT_NB.search(prose_avant[-150:]):
            continue
        s2, e2 = phrase_bounds(texte, a['start'])
        voisins = []
        for b in annotations_with_span(texte):
            if b['start'] == a['start'] or not (s2 <= b['start'] < e2):
                continue
            voisins += re.findall(r'\d+(?:\.\d+)?', b['value'])
        span, motif = span_lettres(prose_avant, a['value'], voisins)
        ligne = {'specie': obj.get('specie'), 'file': os.path.basename(path),
                 'entity': a['entity'], 'avant': texte[a['start']:a['end']],
                 'prose': re.sub(r'\s+', ' ', prose_avant)[-160:]}
        if span is None:
            laisses.append({**ligne, 'motif': motif})
        else:
            modifs.append((a, span))
            faits.append({**ligne, 'apres': f"[{a['entity']} = {span}]"})
    if modifs and APPLIQUER_16:
        for a, span in sorted(modifs, key=lambda x: -x[0]['start']):
            texte = texte[:a['start']] + f"[{a['entity']} = {span}]" + texte[a['end']:]
        obj['corrected_text_final'] = texte
        with open(path, 'w') as fh:
            json.dump(obj, fh, ensure_ascii=False, indent=2)

fa, la = pd.DataFrame(faits), pd.DataFrame(laisses)
print('MODE :', 'APPLIQUE' if APPLIQUER_16 else 'SIMULATION')
print('cas valeur_en_lettres examines :', len(fa) + len(la))
print('  corriges par script          :', len(fa))
print('  laisses au modele            :', len(la))
if len(la):
    print(); print(la.motif.value_counts().to_string())
if len(fa):
    fa.to_csv(os.path.join(OUT_DIR, 'lettres_corriges_script.csv'), index=False)
if len(la):
    la.to_csv(os.path.join(OUT_DIR, 'lettres_pour_llm.csv'), index=False)
fa.head(15)

MODE : APPLIQUE
cas valeur_en_lettres examines : 164
  corriges par script          : 52
  laisses au modele            : 112

motif
span de 10 mots                                 9
valeur du span (2) != annotation (1.0)          8
valeur du span (2) != annotation (0.5)          8
span de 11 mots                                 6
valeur du span (1) != annotation (2.0)          6
valeur du span (3) != annotation (1.0)          5
span de 12 mots                                 4
valeur du span (5) != annotation (17.5)         3
valeur du span (4) != annotation (1.0)          3
valeur du span (1) != annotation (365.0)        3
valeur du span (1) != annotation (4.0)          2
valeur du span (5) != annotation (2.0)          2
valeur du span (12) != annotation (1.0)         2
span de 22 mots                                 2
span de 15 mots                                 2
valeur du span (1) != annotation (3.0)          2
span de 14 mots                                 2
span contient 2, 

,specie,file,entity,avant,prose,apres
0,Alburnoides_eichwaldii_Kaboudval,0190_Alburnoides_eichwaldii_Kaboudval_abj.json,tR,[tR = 2 years after hatching],"At 16 °C , males achieve puberty at one year ...",[tR = two years after hatching]
1,Aotus_trivirgatus,0449_Aotus_trivirgatus_stx.json,F_R_i,[F_R_i = 1.36 litters per year],This low per-event fecundity is coupled with ...,[F_R_i = just over one litter per year]
2,Blepsias_cirrhosus,0799_Blepsias_cirrhosus_abj.json,F_R_i,[F_R_i = approximately 1.1 spawning events per...,earchers first determined that females produce...,[F_R_i = roughly one spawning event per year]
3,Centrarchus_macropterus,1170_Centrarchus_macropterus_abj.json,tp,[tp = approximately 3 years after hatching],The total length at puberty is approximately ...,[tp = around three years after hatching]
4,Centroscymnus_owstonii,1190_Centroscymnus_owstonii_std.json,F_R_i,[F_R_i = approximately 0.5 litters per year],"ecundity and infrequent breeding, with females...",[F_R_i = single litter every 2 years or more]
5,Cephalophus_niger,1209_Cephalophus_niger_stx.json,F_R_i,[F_R_i = 1 calf per year],"ve populations at 39.4 °C , is characterized b...",[F_R_i = approximately one year]
6,Ceriodaphnia_lacustris,1246_Ceriodaphnia_lacustris_std.json,F_R_i,[F_R_i = 1 clutch per instar],"elevated temperature of 25 °C , the species d...",[F_R_i = one clutch produced per instar]
7,Ceriodaphnia_lacustris,1246_Ceriodaphnia_lacustris_std.json,F_R_i,[F_R_i = 1 clutch per instar],its exactly 2 eggs into the brood chamber foll...,[F_R_i = one clutch per instar]
8,Chlorochrysa_calliparaea,1417_Chlorochrysa_calliparaea_std.json,N_R_i,[N_R_i = 2 eggs per clutch],productive biology in the Orange-eared tanager...,[N_R_i = two eggs each year]
9,Chrysichthys_walkeri,1468_Chrysichthys_walkeri_abj.json,F_R_i,[F_R_i = 1 clutch per spawning event],Each gravid female produces a single clutch p...,[F_R_i = single clutch per spawning event]


---
# 17. Corrections validées à la main (ou par modèle)

Pilotée par `corrections_manuelles.csv`, une ligne par décision :

| colonne | contenu |
|---|---|
| `file` | fichier de `species_non_conformes_json/` |
| `action` | `reannoter`, `supprimer` ou `reformuler` |
| `cible` | le texte exact à remplacer (doit apparaître **une seule fois**) |
| `remplacement` | le texte de remplacement |
| `note` | justification, pour la trace |

**Garde-fous** : la cible doit être présente et unique dans le fichier ; après
application, aucune annotation de la phrase touchée ne doit être non conforme.
Toute ligne qui échoue est rejetée sans rien écrire.

In [23]:
TABLE = os.path.join(OUT_DIR, 'corrections_manuelles.csv')

if not os.path.exists(TABLE):
    print('aucun', TABLE)
else:
    tab = pd.read_csv(TABLE)
    faits, rejets = [], []
    for fichier, grp in tab.groupby('file'):
        path = os.path.join(DEST, fichier)
        if not os.path.exists(path):
            rejets += [{**dict(r), 'motif': 'fichier absent'} for _, r in grp.iterrows()]; continue
        obj = json.load(open(path)); texte = obj['corrected_text_final']; avant_tout = texte
        for _, r in grp.iterrows():
            cible, remp = str(r['cible']), str(r['remplacement'])
            n = texte.count(cible)
            if n != 1:
                rejets.append({**dict(r), 'motif': f'cible trouvee {n} fois'}); continue
            texte = texte.replace(cible, remp)
            faits.append(dict(r))
        # verification : la zone modifiee ne doit plus contenir de non-conformite
        restant = [a['entity'] for a in annotations_with_span(texte)
                   if not norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(a['value']))]
        avant_n = len([a for a in annotations_with_span(avant_tout)
                       if not norm_txt(avant_tout[:a['start']].rstrip()).endswith(norm_txt(a['value']))])
        if len(restant) > avant_n:
            rejets.append({'file': fichier, 'motif': 'la correction aggrave le fichier - annulee'})
            continue
        obj['corrected_text_final'] = texte
        with open(path, 'w') as fh:
            json.dump(obj, fh, ensure_ascii=False, indent=2)
    print('corrections appliquees :', len(faits))
    print('rejetees               :', len(rejets))
    for x in rejets:
        print('   ', x.get('file'), '|', x.get('motif'))
    if faits:
        pd.DataFrame(faits).to_csv(os.path.join(OUT_DIR, 'reformulations_faites.csv'), index=False)

corrections appliquees : 100
rejetees               : 6
    1133_Cataetyx_alleni_std.json | cible trouvee 2 fois
    2718_Genypterus_capensis_std.json | cible trouvee 2 fois
    3482_Lepidion_schmidti_std.json | cible trouvee 2 fois
    3650_Lipotes_vexillifer_stx.json | cible trouvee 2 fois
    3650_Lipotes_vexillifer_stx.json | cible trouvee 2 fois
    4263_Mustelus_asterias_std.json | cible trouvee 2 fois


---
# 18. Corrections repérées par position

Piloté par `corrections_position.csv` : `file`, `pos` (offset de début de l'annotation),
`action` (`reannoter` / `supprimer`), `valeur` (le nouveau span, repris de la prose).

Le repérage par position supprime toute ambiguïté de recherche textuelle. Les corrections
d'un même fichier sont appliquées **de la fin vers le début**, pour que les offsets restent
valides.

**Garde-fou** : après une réannotation, la prose qui précède le crochet doit se terminer
par la nouvelle valeur. Sinon la ligne est rejetée et le fichier n'est pas modifié.

In [24]:
TABLE_POS = os.path.join(OUT_DIR, 'corrections_position.csv')

tab = pd.read_csv(TABLE_POS, keep_default_na=False)
faits, rejets = [], []
for fichier, grp in tab.groupby('file'):
    path = os.path.join(DEST, fichier)
    if not os.path.exists(path):
        rejets += [{'file': fichier, 'pos': r['pos'], 'motif': 'fichier absent'} for _, r in grp.iterrows()]
        continue
    obj = json.load(open(path)); texte = obj['corrected_text_final']
    for _, r in grp.sort_values('pos', ascending=False).iterrows():
        pos = int(r['pos'])
        cible = next((a for a in annotations_with_span(texte) if a['start'] == pos), None)
        if cible is None:
            rejets.append({'file': fichier, 'pos': pos, 'motif': 'aucune annotation a cette position'}); continue
        if r['action'] == 'supprimer':
            texte = texte[:cible['start']].rstrip() + texte[cible['end']:]
            faits.append({'file': fichier, 'pos': pos, 'action': 'supprimer',
                          'entity': cible['entity'], 'avant': texte[:0] or cible['value'][:80], 'apres': ''})
            continue
        span = str(r['valeur']).strip()
        if not norm_txt(texte[:cible['start']].rstrip()).endswith(norm_txt(span)):
            reel = re.sub(r'\s+', ' ', texte[max(0, cible['start']-90):cible['start']]).strip()
            rejets.append({'file': fichier, 'pos': pos, 'entity': cible['entity'],
                           'motif': 'span absent de la prose', 'span_propose': span,
                           'prose_reelle': reel[-90:]}); continue
        nouveau = f"[{cible['entity']} = {span}]"
        faits.append({'file': fichier, 'pos': pos, 'action': 'reannoter',
                      'entity': cible['entity'], 'avant': cible['value'][:80], 'apres': span[:80]})
        texte = texte[:cible['start']] + nouveau + texte[cible['end']:]
    obj['corrected_text_final'] = texte
    with open(path, 'w') as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2)

print('appliquees :', len(faits), '| rejetees :', len(rejets))
if faits:
    pd.DataFrame(faits).to_csv(os.path.join(OUT_DIR, 'corrections_position_faites.csv'), index=False)
    print(); print(pd.DataFrame(faits).action.value_counts().to_string())
if rejets:
    rj = pd.DataFrame(rejets)
    rj.to_csv(os.path.join(OUT_DIR, 'corrections_position_rejets.csv'), index=False)
    print(); print(rj.motif.value_counts().to_string())

appliquees : 67 | rejetees : 32

action
reannoter    39
supprimer    28

motif
aucune annotation a cette position    32


---
# 19. Suppression des annotations `T` sans température dans la phrase

Le prompt de génération demande à la fois d'annoter `T` à chaque mesure (ligne 58) et
de ne pas répéter une température partagée (ligne 61). Le modèle a résolu le conflit en
posant l'annotation après une périphrase :

```
... lasting 183 days under the same thermal regime [T = 10 °C]
```

Le span désigné n'est pas une valeur de température. Ces annotations ne produisent
d'ailleurs aucun span BIO. **Critère de suppression, sans ambiguïté** : entité `T`, et
aucune occurrence de `nombre °C` dans la phrase.

In [25]:
TEMP_VAL = re.compile(r'\d+(?:\.\d+)?\s*°C')

supprimees, fichiers = [], 0
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    cibles = []
    annots = annotations_with_span(texte)
    for a in annots:
        if a['entity'] != 'T' or a['nested']:
            continue
        if norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(a['value'])):
            continue                                   # conforme : on ne touche pas
        s, e = phrase_bounds(texte, a['start'])
        prose, _ = prose_seule(texte, s, e, annots)
        if TEMP_VAL.search(prose):
            continue                                   # une temperature figure ailleurs : a deplacer, pas a supprimer
        cibles.append(a)
    if not cibles:
        continue
    fichiers += 1
    for a in sorted(cibles, key=lambda x: -x['start']):
        supprimees.append({'specie': obj.get('specie'), 'file': os.path.basename(path),
                           'valeur': a['value'],
                           'prose': re.sub(r'\s+', ' ', texte[max(0, a['start']-90):a['start']]).strip()[-80:]})
        texte = texte[:a['start']].rstrip() + texte[a['end']:]
    obj['corrected_text_final'] = texte
    with open(path, 'w') as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2)

sp = pd.DataFrame(supprimees)
print('annotations T supprimees :', len(sp), '| fichiers :', fichiers)
if len(sp):
    sp.to_csv(os.path.join(OUT_DIR, 'T_supprimees.csv'), index=False)
    print(); print(sp.head(6).to_string(index=False))

annotations T supprimees : 96 | fichiers : 88

                specie                                 file  valeur                                                                            prose
   Abralia_multihamata    0005_Abralia_multihamata_abj.json   10 °C span for females, lasting 183 days [am = 183 days] under the same thermal regime
         Abramis_brama          0008_Abramis_brama_abj.json   18 °C atching [tp = around 730 days after hatching] under identical thermal conditions
  Acipenser_oxyrinchus   0080_Acipenser_oxyrinchus_std.json   17 °C  since birth [tp = roughly 7 years since birth] under similar thermal conditions
   Aerodramus_bartschi    0134_Aerodramus_bartschi_std.json 38.1 °C pproximately 7702 days], or just over 21 years under the same thermal conditions
      Alosa_aestivalis       0242_Alosa_aestivalis_abj.json   17 °C at about 3.7 years [tp = about 3.7 years], again within this same thermal regime
Antidorcas_marsupialis 0437_Antidorcas_marsupialis_stx.json

---
# 20. Corrections finales, repérées par position

Placée **en fin de pipeline** : les positions y sont mesurées sur l'état final du dossier,
après toutes les sections précédentes. C'est la table à utiliser quand une correction ne
peut pas être ciblée par le texte — typiquement lorsqu'une même annotation apparaît deux
fois dans un fichier et que seule l'une des deux doit changer.

Pilotée par `corrections_finales.csv` : `file`, `pos`, `action`, `valeur`.

In [26]:
TABLE_FIN = os.path.join(OUT_DIR, 'corrections_finales.csv')

if not os.path.exists(TABLE_FIN):
    print('aucun', TABLE_FIN)
else:
    tab = pd.read_csv(TABLE_FIN, keep_default_na=False)
    faits, rejets = [], []
    for fichier, grp in tab.groupby('file'):
        path = os.path.join(DEST, fichier)
        obj = json.load(open(path)); texte = obj['corrected_text_final']
        for _, r in grp.sort_values('pos', ascending=False).iterrows():
            pos = int(r['pos'])
            a = next((x for x in annotations_with_span(texte) if x['start'] == pos), None)
            if a is None:
                rejets.append({'file': fichier, 'pos': pos, 'motif': 'aucune annotation a cette position'}); continue
            if r['action'] == 'supprimer':
                texte = texte[:a['start']].rstrip() + texte[a['end']:]
                faits.append({'file': fichier, 'pos': pos, 'action': 'supprimer', 'entity': a['entity']})
            else:
                span = str(r['valeur']).strip()
                if not norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(span)):
                    rejets.append({'file': fichier, 'pos': pos, 'motif': 'span absent de la prose'}); continue
                texte = texte[:a['start']] + f"[{a['entity']} = {span}]" + texte[a['end']:]
                faits.append({'file': fichier, 'pos': pos, 'action': 'reannoter', 'entity': a['entity']})
        obj['corrected_text_final'] = texte
        with open(path, 'w') as fh:
            json.dump(obj, fh, ensure_ascii=False, indent=2)
    print('appliquees :', len(faits), '| rejetees :', len(rejets))
    for x in rejets:
        print('   ', x['file'], x['pos'], '|', x['motif'])

appliquees : 5 | rejetees : 0


---
# 21. Seconde passe de relocalisation

Les sections 16 à 20 modifient le texte, ce qui crée de nouveaux cas où la valeur figure
dans la prose mais où le crochet n'est plus adjacent. Cette passe rejoue la relocalisation
de la section 13 sur l'état final : le crochet vient se placer juste après la dernière
occurrence de sa valeur dans la phrase, annotations exclues.

Aucune perte possible : seule la position du crochet change.

In [ ]:
def motif_valeur(v):
    """Regex tolerante aux espaces (motif() est masquee par une variable en section 16)."""
    mots = [re.escape(w) for w in v.split()]
    return re.compile(r'\s*'.join(mots), re.I) if mots else None

faits2 = []
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    change = True
    while change:                       # une annotation a la fois : les offsets bougent
        change = False
        annots = annotations_with_span(texte)
        for a in annots:
            if a['nested'] or norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(a['value'])):
                continue
            s, e = phrase_bounds(texte, a['start'])
            prose, idx = prose_seule(texte, s, e, annots)
            pat = motif_valeur(a['value'])
            cible = None
            if pat:
                for m in pat.finditer(prose):
                    cible = m
            if cible is None or cible.end() > len(idx):
                continue
            ins = idx[cible.end() - 1] + 1
            bloc = texte[a['start']:a['end']]
            if ins < a['start']:
                texte = (texte[:ins] + ' ' + bloc
                         + texte[ins:a['start']].rstrip() + texte[a['end']:])
            else:
                texte = (texte[:a['start']].rstrip() + texte[a['end']:ins]
                         + ' ' + bloc + texte[ins:])
            faits2.append({'file': os.path.basename(path), 'entity': a['entity'],
                           'valeur': a['value'][:70]})
            change = True
            break
    obj['corrected_text_final'] = texte
    with open(path, 'w') as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2)

print('crochets deplaces :', len(faits2))
if faits2:
    f2 = pd.DataFrame(faits2)
    print(); print(f2.entity.value_counts().to_string())
    f2.to_csv(os.path.join(OUT_DIR, 'relocalisations_passe2.csv'), index=False)

---
# 22. Suppression des doublons stricts

Deux annotations **identiques** (même entité, même valeur) posées l'une contre l'autre,
sans texte entre elles :

```
... recorded exclusively at 17.9 °C [T = 17.9 °C] [T = 17.9 °C], a temperature that ...
```

La seconde est redondante et non conforme — le texte qui la précède se termine par un
crochet, pas par une valeur. On la supprime.

**Ne sont pas concernées** les annotations accolées d'entités *différentes* : chez
*Cyclops vicinus* `[Li = 0.1627 cm] [Lp = 0.1627 cm]`, deux traits distincts valent
réellement la même chose et la phrase l'énonce. Elles sont légitimes et restent en place.

In [ ]:
doublons = []
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    change = True
    while change:
        change = False
        annots = annotations_with_span(texte)
        for k in range(1, len(annots)):
            a, prec = annots[k], annots[k-1]
            if texte[prec['end']:a['start']].strip():
                continue                               # du texte entre les deux
            if a['entity'] != prec['entity']:
                continue                               # entites differentes : legitime
            if norm_txt(a['value']) != norm_txt(prec['value']):
                continue                               # meme entite, valeurs differentes
            doublons.append({'specie': obj.get('specie'), 'file': os.path.basename(path),
                             'entity': a['entity'], 'valeur': a['value'][:60]})
            texte = texte[:a['start']].rstrip() + texte[a['end']:]
            change = True
            break
    obj['corrected_text_final'] = texte
    with open(path, 'w') as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2)

db = pd.DataFrame(doublons)
print('doublons supprimes :', len(db))
if len(db):
    print(); print(db.entity.value_counts().to_string())
    db.to_csv(os.path.join(OUT_DIR, 'doublons_supprimes.csv'), index=False)

---
# 23. Aligner l'annotation sur la prose

Règle : **l'annotation reprend le texte tel qu'il est écrit.**

```
prose : range between 52 and 64 grams   annotation : [Ww0 = 52-64 grams]
apres : range between 52 and 64 grams [Ww0 = between 52 and 64 grams]
```

On remonte mot à mot depuis le crochet jusqu'à obtenir un fragment contenant **tous les
nombres de la valeur annotée**, puis on étend à gauche sur les modificateurs
(`between`, `from`, `approximately`, `around`, `roughly`, `about`, `nearly`, `up to`).

**Garde-fou décisif** : les nombres du nouveau fragment doivent être *exactement* ceux de
l'ancienne valeur — ni plus, ni moins. La valeur ne peut donc pas changer, seule sa
formulation le peut. Un fragment qui happerait le chiffre d'une entité voisine est rejeté.

Bornes : 2 à 12 mots, aucun crochet dans le fragment.

In [ ]:
NOMBRE = re.compile(r'\d+(?:\.\d+)?')
MODIF_G = ('between','from','approximately','around','roughly','about','nearly','up','to',
           'over','under','at','a','an','the','of','some','least','most','than','just')

alignes, echecs = [], []
for path in sorted(glob.glob(DEST + '/*.json')):
    obj = json.load(open(path))
    if isinstance(obj, list):
        continue
    texte = obj.get('corrected_text_final') or ''
    annots = annotations_with_span(texte)
    todo = []
    for a in annots:
        if a['nested'] or norm_txt(texte[:a['start']].rstrip()).endswith(norm_txt(a['value'])):
            continue
        ref = set(NOMBRE.findall(a['value']))
        if not ref:
            echecs.append({'specie': obj.get('specie'), 'entity': a['entity'],
                           'valeur': a['value'][:70], 'motif': 'valeur sans nombre'}); continue
        s, e = phrase_bounds(texte, a['start'])
        prose, _ = prose_seule(texte, s, a['start'], annots)
        mots = prose.split()
        k = None
        for n in range(1, min(len(mots), 12) + 1):
            if set(NOMBRE.findall(' '.join(mots[-n:]))) >= ref:
                k = n; break
        if k is None:
            echecs.append({'specie': obj.get('specie'), 'entity': a['entity'],
                           'valeur': a['value'][:70], 'motif': 'nombres absents de la prose'}); continue
        while k < 12 and k < len(mots) and mots[-(k+1)].lower().strip(',;:') in MODIF_G:
            k += 1
        cand = ' '.join(mots[-k:]).strip(' ,;:')
        motif = None
        if '[' in cand or ']' in cand:               motif = 'fragment contenant un crochet'
        elif not (2 <= len(cand.split()) <= 12):     motif = f'fragment de {len(cand.split())} mots'
        elif set(NOMBRE.findall(cand)) != ref:       motif = 'nombres differents de la valeur'
        elif not norm_txt(prose).endswith(norm_txt(cand)): motif = 'fragment non terminal'
        if motif:
            echecs.append({'specie': obj.get('specie'), 'entity': a['entity'],
                           'valeur': a['value'][:70], 'motif': motif}); continue
        todo.append((a, cand))
    if not todo:
        continue
    for a, cand in sorted(todo, key=lambda x: -x[0]['start']):
        alignes.append({'specie': obj.get('specie'), 'file': os.path.basename(path),
                        'entity': a['entity'], 'avant': a['value'][:70], 'apres': cand[:70]})
        texte = texte[:a['start']] + f"[{a['entity']} = {cand}]" + texte[a['end']:]
    obj['corrected_text_final'] = texte
    with open(path, 'w') as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2)

al, ec = pd.DataFrame(alignes), pd.DataFrame(echecs)
print('annotations alignees sur la prose :', len(al))
print('non traitees                      :', len(ec))
if len(al):
    print(); print(al.entity.value_counts().to_string())
    al.to_csv(os.path.join(OUT_DIR, 'alignees_sur_prose.csv'), index=False)
if len(ec):
    print(); print(ec.motif.value_counts().to_string())
    ec.to_csv(os.path.join(OUT_DIR, 'non_alignables.csv'), index=False)